In [54]:
import numpy as np
import pandas as pd



In [55]:
df = pd.read_csv('Titanic-Dataset.csv')
print(df.shape)
df.head()
#checking for null and missing values
df.info()
print()
df.isnull().sum()


(891, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB



PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [56]:
#age has 177 missing values, cabin has 687 missing values, embarked has 2 missing values. We will drop the cabin column as it has too many missing values. We will fill the missing values in age with the median value of age and we will fill the missing values in embarked with the mode value of embarked.
#EDA-checking survival rate by sex. and this code split sex into male female 
print(df.groupby('Sex')['Survived'].mean())
print()
print(df.groupby('Pclass')['Survived'].mean())


Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64

Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64


In [57]:
#cleaning the data , encode categorical coulumns + add Family size
df['Sex']=df['Sex'].map({'male':0,'female':1})
df['Embarked']=df['Embarked'].map({'S':0,'C':1,'Q':2})
df['FamilySize']=df['SibSp']+df['Parch'] + 1
df['IsAlone']=(df['FamilySize'] ==1).astype(int)


In [58]:
df.drop(columns=['PassengerId', 'Name', 'Ticket'], inplace=True)#removes three columns permanently from df:

X = df.drop(columns=['Survived'])
y = df['Survived']

X.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked,FamilySize,IsAlone
0,3,0,22.0,1,0,7.2500,NaN,0.0,2,0
1,1,1,38.0,1,0,71.2833,C85,1.0,2,0
2,3,1,26.0,0,0,7.9250,NaN,0.0,1,1
3,1,1,35.0,1,0,53.1000,C123,0.0,2,0
4,3,0,35.0,0,0,8.0500,NaN,0.0,1,1


In [59]:
#train test split
from sklearn.model_selection import train_test_split 

X_train,X_test, y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
print(X_train.shape,X_test.shape )


(712, 10) (179, 10)


In [64]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# drop Cabin before splitting so it never comes back
X_clean = X.drop(columns=['Cabin'])

X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y, test_size=0.2, random_state=42
)

# fill any missing values using training data stats only
for col in X_train.columns:
    if X_train[col].isnull().sum() > 0:
        fill_value = X_train[col].median()
        X_train[col] = X_train[col].fillna(fill_value)
        X_test[col] = X_test[col].fillna(fill_value)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy_score(y_test, y_pred)

0.7988826815642458

In [65]:
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(X_train,y_train)
rf_pred =rf_model.predict(X_test)
accuracy_score(y_test,rf_pred)

0.8156424581005587

In [ ]:
import pandas as pd 
importances =pd.Series(rf_model.feature_importances_,index=X_train.columns)#Random forest model has a feature_importances_ attribute that can be used to get the importance of each feature in the model. We can use this attribute to create a pandas Series object that contains the feature importances and their corresponding feature names.
importances.sort_values(ascending=False)#sorts the list into highest to lowest

Sex           0.280463
Fare          0.255974
Age           0.244345
Pclass        0.084678
FamilySize    0.043566
Embarked      0.032155
SibSp         0.028804
Parch         0.020261
IsAlone       0.009754
dtype: float64

In [67]:
#cross validation
from sklearn.model_selection import cross_val_score
cv_scores = cross_val_score(rf_model,X_clean,y,cv=5)
print("Cross-validation scores:", cv_scores)
print("Mean cross-validation score:", cv_scores.mean())


Cross-validation scores: [0.77653631 0.80337079 0.85955056 0.7752809  0.84269663]
Mean cross-validation score: 0.8114870378507313


In [68]:
#writing actual submission/output file 
predictions_df =pd.DataFrame({
    'Actual':y_test.values,
    'Predictions':rf_model.predict(X_test)
})
predictions_df.to_csv('Titanic-Dataset.csv', index=False)
predictions_df.head()


,Actual,Predictions
0,1,0
1,0,0
2,0,0
3,1,1
4,1,0
